In [1]:
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

SOURCE_URL = "https://www.serebii.net/pokemon/nationalpokedex.shtml"
OUTPUT_FILE = Path("pokedex_serebii.csv")

REQUEST_HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; PokedexExplorer/1.0; educational project)"
}

POKEMON_TYPES = {
    "normal", "fire", "water", "electric", "grass", "ice",
    "fighting", "poison", "ground", "flying", "psychic", "bug",
    "rock", "ghost", "dragon", "dark", "steel", "fairy",
}

PROJECT_COLUMNS = [
    "Pokemon", "Generation", "HP", "Attack", "Defense",
    "Special_Attack", "Special_Defense", "Speed",
    "hp_over_total", "attack_over_total", "defense_over_total",
    "SPA_over_total", "SPD_over_total", "speed_over_total", "Total",
    "Type_I", "Type II", "Ability_I", "Ability_II", "Hidden_Ability",
    "Height_m", "Weight_kg", "Height_over_weight", "HP_EV", "Attcak_EV",
    "Defense_EV", "Special_Attack_EV", "Special_Defense_EV", "Speed_EV",
]

session = requests.Session()
session.headers.update(REQUEST_HEADERS)

def generation_for_dex_number(dex_number):
    generation_limits = [151, 251, 386, 493, 649, 721, 809, 905, 1025]
    for generation_number, last_number in enumerate(generation_limits, start=1):
        if dex_number <= last_number:
            return generation_number
    return 9

def normalize_type_label(value):
    """Normalize a label or image filename and return a valid Pokémon type, if found."""
    normalized = re.sub(r"[^a-z]", "", str(value).lower())
    return normalized.title() if normalized in POKEMON_TYPES else None

def extract_types(type_cell):
    """Read types from icon alt/title text, icon filenames, and linked text."""
    found_types = []

    for image_tag in type_cell.find_all("img"):
        candidates = [
            image_tag.get("alt", ""),
            image_tag.get("title", ""),
            image_tag.get("src", "").split("/")[-1].split(".")[0],
        ]
        for candidate in candidates:
            pokemon_type = normalize_type_label(candidate)
            if pokemon_type and pokemon_type not in found_types:
                found_types.append(pokemon_type)
                break

    for link in type_cell.find_all("a"):
        pokemon_type = normalize_type_label(link.get_text(" ", strip=True))
        if pokemon_type and pokemon_type not in found_types:
            found_types.append(pokemon_type)

    # Also inspect the cell's visible text, in case types are plain text.
    for token in re.split(r"[^A-Za-z]+", type_cell.get_text(" ", strip=True)):
        pokemon_type = normalize_type_label(token)
        if pokemon_type and pokemon_type not in found_types:
            found_types.append(pokemon_type)

    return found_types[:2]

def extract_height_metres(text):
    """Extract a height in metres from a Serebii measurement string."""
    match = re.search(r"(\d+(?:\.\d+)?)\s*m\b", text, flags=re.IGNORECASE)
    return float(match.group(1)) if match else None

def extract_weight_kg(text):
    """Extract a weight in kilograms from a Serebii measurement string."""
    match = re.search(r"(\d+(?:\.\d+)?)\s*kg\b", text, flags=re.IGNORECASE)
    return float(match.group(1)) if match else None

def extract_measurement_near_label(soup, label_pattern):
    """Find a table cell labelled Height or Weight and read its adjacent value."""
    pattern = re.compile(label_pattern, flags=re.IGNORECASE)

    for cell in soup.find_all(["td", "th"]):
        cell_label = cell.get_text(" ", strip=True).strip(" :")
        if pattern.fullmatch(cell_label):
            # Prefer the next cell in the same row.
            parent_row = cell.find_parent("tr")
            if parent_row:
                row_cells = parent_row.find_all(["td", "th"], recursive=False)
                try:
                    cell_index = row_cells.index(cell)
                except ValueError:
                    cell_index = -1

                if cell_index >= 0 and cell_index + 1 < len(row_cells):
                    value_text = row_cells[cell_index + 1].get_text(" ", strip=True)
                    if value_text:
                        return value_text

            # Fallback: inspect the next sibling cell.
            sibling = cell.find_next(["td", "th"])
            if sibling:
                return sibling.get_text(" ", strip=True)

    return ""

def scrape_measurements(detail_url):
    """Download one Pokémon detail page and return height in metres and weight in kg."""
    if not detail_url:
        return None, None

    try:
        response = session.get(detail_url, timeout=25)
        response.raise_for_status()
    except requests.RequestException as error:
        print(f"Could not fetch detail page {detail_url}: {error}")
        return None, None

    detail_soup = BeautifulSoup(response.text, "html.parser")
    height_text = extract_measurement_near_label(detail_soup, r"Height")
    weight_text = extract_measurement_near_label(detail_soup, r"Weight")

    return extract_height_metres(height_text), extract_weight_kg(weight_text)

print("Ready. CSV output will be saved to:", OUTPUT_FILE.resolve())

Ready. CSV output will be saved to: C:\Users\tykur\pokedex_serebii.csv


In [2]:
# Download the National Pokédex index page.
response = session.get(SOURCE_URL, timeout=30)
response.raise_for_status()

pokedex_soup = BeautifulSoup(response.text, "html.parser")

print("Page title:", pokedex_soup.title.get_text(" ", strip=True) if pokedex_soup.title else "(no title)")
print("Tables found:", len(pokedex_soup.find_all("table")))

Page title: Pokémon - National Pokédex Pokémon
Tables found: 1026


In [3]:
def parse_pokemon_row(row):
    cells = row.find_all("td", recursive=False)
    if not cells:
        return None

    row_text = row.get_text(" ", strip=True)
    dex_match = re.search(r"#\s*0*(\d{1,4})", row_text)
    if not dex_match:
        return None

    dex_number = int(dex_match.group(1))

    # Find the Pokémon's name and its detail-page URL.
    pokemon_name = ""
    detail_url = ""
    for cell in cells:
        for anchor in cell.find_all("a", href=True):
            anchor_text = anchor.get_text(" ", strip=True)
            href = anchor["href"]
            if anchor_text and not anchor_text.startswith("#"):
                pokemon_name = anchor_text
                detail_url = urljoin(SOURCE_URL, href)
                break
        if pokemon_name:
            break

    if not pokemon_name:
        return None

    # Find type information in cells containing type icons or type labels.
    all_types = []
    for cell in cells:
        for pokemon_type in extract_types(cell):
            if pokemon_type not in all_types:
                all_types.append(pokemon_type)

    # Collect numeric-only cells. The base-stat table generally includes six
    # stats followed by a total, so use the six values immediately before total
    # when seven or more numeric cells are present.
    numeric_cells = []
    for cell in cells:
        cell_text = cell.get_text(" ", strip=True).replace(",", "")
        if re.fullmatch(r"\d{1,4}", cell_text):
            numeric_cells.append(int(cell_text))

    if len(numeric_cells) >= 7:
        stats = numeric_cells[-7:-1]
    elif len(numeric_cells) >= 6:
        stats = numeric_cells[-6:]
    else:
        return None

    hp, attack, defense, special_attack, special_defense, speed = stats
    total = hp + attack + defense + special_attack + special_defense + speed

    # Keep linked ability names, excluding the Pokémon name and non-ability links.
    abilities = []
    for cell in cells:
        for anchor in cell.find_all("a", href=True):
            label = anchor.get_text(" ", strip=True)
            href = anchor["href"].lower()
            if (
                label
                and label != pokemon_name
                and not label.startswith("#")
                and ("ability" in href or "ability" in label.lower())
                and label not in abilities
            ):
                abilities.append(label)

    return {
        "_dex_number": dex_number,
        "Pokemon": pokemon_name,
        "Generation": generation_for_dex_number(dex_number),
        "HP": hp,
        "Attack": attack,
        "Defense": defense,
        "Special_Attack": special_attack,
        "Special_Defense": special_defense,
        "Speed": speed,
        "hp_over_total": hp / total if total else None,
        "attack_over_total": attack / total if total else None,
        "defense_over_total": defense / total if total else None,
        "SPA_over_total": special_attack / total if total else None,
        "SPD_over_total": special_defense / total if total else None,
        "speed_over_total": speed / total if total else None,
        "Total": total,
        "Type_I": all_types[0] if all_types else "",
        "Type II": all_types[1] if len(all_types) > 1 else "",
        "Ability_I": abilities[0] if abilities else "",
        "Ability_II": abilities[1] if len(abilities) > 1 else "",
        "Hidden_Ability": abilities[2] if len(abilities) > 2 else "",
        "Height_m": None,
        "Weight_kg": None,
        "Height_over_weight": None,
        "HP_EV": None,
        "Attcak_EV": None,
        "Defense_EV": None,
        "Special_Attack_EV": None,
        "Special_Defense_EV": None,
        "Speed_EV": None,
        "_detail_url": detail_url,
    }

parsed_rows = []
for table_row in pokedex_soup.find_all("tr"):
    parsed_pokemon = parse_pokemon_row(table_row)
    if parsed_pokemon:
        parsed_rows.append(parsed_pokemon)

# Deduplicate by National Pokédex number.
unique_rows = {}
for pokemon in parsed_rows:
    unique_rows[pokemon["_dex_number"]] = pokemon

pokemon_records = [unique_rows[number] for number in sorted(unique_rows)]
print("Unique Pokémon rows parsed:", len(pokemon_records))

if not pokemon_records:
    raise ValueError(
        "No Pokémon rows were parsed. Serebii may have changed the page structure. "
        "Inspect the HTML and update parse_pokemon_row()."
    )

display(pd.DataFrame(pokemon_records).head(10)[
    ["_dex_number", "Pokemon", "Type_I", "Type II", "HP", "Attack", "Defense",
     "Special_Attack", "Special_Defense", "Speed", "_detail_url"]
])

Unique Pokémon rows parsed: 1025


,_dex_number,Pokemon,Type_I,Type II,HP,Attack,Defense,Special_Attack,Special_Defense,Speed,_detail_url
0,1,Bulbasaur,Grass,Poison,45,49,49,65,65,45,https://www.serebii.net/pokemon/bulbasaur
1,2,Ivysaur,Grass,Poison,60,62,63,80,80,60,https://www.serebii.net/pokemon/ivysaur
2,3,Venusaur,Grass,Poison,80,82,83,100,100,80,https://www.serebii.net/pokemon/venusaur
3,4,Charmander,Fire,,39,52,43,60,50,65,https://www.serebii.net/pokemon/charmander
4,5,Charmeleon,Fire,,58,64,58,80,65,80,https://www.serebii.net/pokemon/charmeleon
5,6,Charizard,Fire,Flying,78,84,78,109,85,100,https://www.serebii.net/pokemon/charizard
6,7,Squirtle,Water,,44,48,65,50,64,43,https://www.serebii.net/pokemon/squirtle
7,8,Wartortle,Water,,59,63,80,65,80,58,https://www.serebii.net/pokemon/wartortle
8,9,Blastoise,Water,,79,83,100,85,105,78,https://www.serebii.net/pokemon/blastoise
9,10,Caterpie,Bug,,45,30,35,20,20,45,https://www.serebii.net/pokemon/caterpie


In [4]:
# Height/weight from Serebii, EVs from PokeAPI (Serebii's detail page has no EV data).
# This can take several minutes. A short delay reduces request load on the sites.

POKEAPI_URL = "https://pokeapi.co/api/v2/pokemon/{}"

EV_COLUMNS = {
    "hp": "HP_EV",
    "attack": "Attcak_EV",  # spelling matches PROJECT_COLUMNS
    "defense": "Defense_EV",
    "special-attack": "Special_Attack_EV",
    "special-defense": "Special_Defense_EV",
    "speed": "Speed_EV",
}

def value_below_header(soup, label):
    """Find a header cell (e.g. 'Height') and return the text in the same column of the next row."""
    for cell in soup.find_all(["td", "th"]):
        if cell.get_text(" ", strip=True).strip(" :").lower() != label.lower():
            continue
        row = cell.find_parent("tr")
        next_row = row.find_next_sibling("tr") if row else None
        if not next_row:
            continue
        header_cells = row.find_all(["td", "th"], recursive=False)
        value_cells = next_row.find_all(["td", "th"], recursive=False)
        if cell in header_cells:
            idx = header_cells.index(cell)
            if idx < len(value_cells):
                return value_cells[idx].get_text(" ", strip=True)
    return ""

def scrape_measurements_v2(detail_url):
    """Return (height_m, weight_kg) from a Serebii detail page."""
    if not detail_url:
        return None, None
    try:
        response = session.get(detail_url, timeout=25)
        response.raise_for_status()
    except requests.RequestException as error:
        print(f"Could not fetch {detail_url}: {error}")
        return None, None

    soup = BeautifulSoup(response.text, "html.parser")
    text = re.sub(r"\s+", " ", soup.get_text(" ", strip=True))
    height = extract_height_metres(value_below_header(soup, "Height")) \
        or extract_height_metres(text)
    weight = extract_weight_kg(value_below_header(soup, "Weight")) \
        or extract_weight_kg(text)
    return height, weight

def fetch_evs(dex_number):
    """Return {column: EV yield} for one Pokemon from PokeAPI."""
    empty = {col: None for col in EV_COLUMNS.values()}
    for attempt in range(3):
        try:
            response = session.get(POKEAPI_URL.format(dex_number), timeout=25)
            response.raise_for_status()
            evs = {col: 0 for col in EV_COLUMNS.values()}
            for stat in response.json()["stats"]:
                column = EV_COLUMNS.get(stat["stat"]["name"])
                if column:
                    evs[column] = stat["effort"]
            return evs
        except (requests.RequestException, KeyError, ValueError) as error:
            if attempt == 2:
                print(f"PokeAPI failed for #{dex_number}: {error}")
            time.sleep(1)
    return empty

# Quick sanity check before the full run (Bulbasaur: 0.7, 6.9, Special_Attack_EV=1).
test = pokemon_records[0]
print("Test:", scrape_measurements_v2(test["_detail_url"]), fetch_evs(test["_dex_number"]))

for index, pokemon in enumerate(pokemon_records, start=1):
    height_m, weight_kg = scrape_measurements_v2(pokemon.get("_detail_url", ""))
    pokemon["Height_m"] = height_m
    pokemon["Weight_kg"] = weight_kg
    pokemon["Height_over_weight"] = (
        height_m / weight_kg if height_m is not None and weight_kg else None
    )
    pokemon.update(fetch_evs(pokemon["_dex_number"]))

    if index % 25 == 0 or index == len(pokemon_records):
        print(f"Processed {index}/{len(pokemon_records)} Pokémon")

    time.sleep(0.25)

print("Finished fetching details.")


Test: (0.7, 6.9) {'HP_EV': 0, 'Attcak_EV': 0, 'Defense_EV': 0, 'Special_Attack_EV': 1, 'Special_Defense_EV': 0, 'Speed_EV': 0}
Processed 25/1025 Pokémon
Processed 50/1025 Pokémon
Processed 75/1025 Pokémon
Processed 100/1025 Pokémon
Processed 125/1025 Pokémon
Processed 150/1025 Pokémon
Processed 175/1025 Pokémon
Processed 200/1025 Pokémon
Processed 225/1025 Pokémon
Processed 250/1025 Pokémon
Processed 275/1025 Pokémon
Processed 300/1025 Pokémon
Processed 325/1025 Pokémon
Processed 350/1025 Pokémon
Processed 375/1025 Pokémon
Processed 400/1025 Pokémon
Processed 425/1025 Pokémon
Processed 450/1025 Pokémon
Processed 475/1025 Pokémon
Processed 500/1025 Pokémon
Processed 525/1025 Pokémon
Processed 550/1025 Pokémon
Processed 575/1025 Pokémon
Processed 600/1025 Pokémon
Processed 625/1025 Pokémon
Processed 650/1025 Pokémon
Processed 675/1025 Pokémon
Processed 700/1025 Pokémon
Processed 725/1025 Pokémon
Processed 750/1025 Pokémon
Processed 775/1025 Pokémon
Processed 800/1025 Pokémon
Processed 82

In [5]:
# Create the final DataFrame and validate key fields.
pokemon_df = pd.DataFrame(pokemon_records)

pokemon_df = pokemon_df.drop(
    columns=["_dex_number", "_detail_url"],
    errors="ignore",
)

pokemon_df = pokemon_df.reindex(columns=PROJECT_COLUMNS)

for column in [
    "HP", "Attack", "Defense", "Special_Attack",
    "Special_Defense", "Speed", "Total",
    "Height_m", "Weight_kg", "Height_over_weight",
    "HP_EV", "Attcak_EV", "Defense_EV",
    "Special_Attack_EV", "Special_Defense_EV", "Speed_EV",
]:
    pokemon_df[column] = pd.to_numeric(pokemon_df[column], errors="coerce")

print("Rows:", len(pokemon_df))
print("Unique names:", pokemon_df["Pokemon"].nunique())
print("Missing primary types:", (pokemon_df["Type_I"].fillna("").str.strip() == "").sum())
print("Missing secondary types:", (pokemon_df["Type II"].fillna("").str.strip() == "").sum())
print("Missing heights:", pokemon_df["Height_m"].isna().sum())
print("Missing weights:", pokemon_df["Weight_kg"].isna().sum())
print("Missing EVs:", pokemon_df["HP_EV"].isna().sum())

display(pokemon_df.head(15))
display(pokemon_df[
    ["Pokemon", "Type_I", "Type II", "Height_m", "Weight_kg"]
].head(25))

Rows: 1025
Unique names: 1025
Missing primary types: 0
Missing secondary types: 489
Missing heights: 0
Missing weights: 0
Missing EVs: 0


,Pokemon,Generation,HP,Attack,Defense,Special_Attack,Special_Defense,Speed,hp_over_total,attack_over_total,...,Hidden_Ability,Height_m,Weight_kg,Height_over_weight,HP_EV,Attcak_EV,Defense_EV,Special_Attack_EV,Special_Defense_EV,Speed_EV
0,Bulbasaur,1,45,49,49,65,65,45,0.141509,0.154088,...,,0.7,6.9,0.101449,0,0,0,1,0,0
1,Ivysaur,1,60,62,63,80,80,60,0.148148,0.153086,...,,1.0,13.0,0.076923,0,0,0,1,1,0
2,Venusaur,1,80,82,83,100,100,80,0.152381,0.156190,...,,2.0,100.0,0.020000,0,0,0,2,1,0
3,Charmander,1,39,52,43,60,50,65,0.126214,0.168285,...,,0.6,8.5,0.070588,0,0,0,0,0,1
4,Charmeleon,1,58,64,58,80,65,80,0.143210,0.158025,...,,1.1,19.0,0.057895,0,0,0,1,0,1
5,Charizard,1,78,84,78,109,85,100,0.146067,0.157303,...,,1.7,90.5,0.018785,0,0,0,3,0,0
6,Squirtle,1,44,48,65,50,64,43,0.140127,0.152866,...,,0.5,9.0,0.055556,0,0,1,0,0,0
7,Wartortle,1,59,63,80,65,80,58,0.145679,0.155556,...,,1.0,22.5,0.044444,0,0,1,0,1,0
8,Blastoise,1,79,83,100,85,105,78,0.149057,0.156604,...,,1.6,85.5,0.018713,0,0,0,0,3,0
9,Caterpie,1,45,30,35,20,20,45,0.230769,0.153846,...,,0.3,2.9,0.103448,1,0,0,0,0,0


,Pokemon,Type_I,Type II,Height_m,Weight_kg
0,Bulbasaur,Grass,Poison,0.7,6.9
1,Ivysaur,Grass,Poison,1.0,13.0
2,Venusaur,Grass,Poison,2.0,100.0
3,Charmander,Fire,,0.6,8.5
4,Charmeleon,Fire,,1.1,19.0
5,Charizard,Fire,Flying,1.7,90.5
6,Squirtle,Water,,0.5,9.0
7,Wartortle,Water,,1.0,22.5
8,Blastoise,Water,,1.6,85.5
9,Caterpie,Bug,,0.3,2.9


In [6]:
# Create the final DataFrame and validate key fields.
pokemon_df = pd.DataFrame(pokemon_records)

pokemon_df = pokemon_df.drop(
    columns=["_dex_number", "_detail_url"],
    errors="ignore",
)

pokemon_df = pokemon_df.reindex(columns=PROJECT_COLUMNS)

for column in [
    "HP", "Attack", "Defense", "Special_Attack",
    "Special_Defense", "Speed", "Total",
    "Height_m", "Weight_kg", "Height_over_weight",
    "HP_EV", "Attcak_EV", "Defense_EV",
    "Special_Attack_EV", "Special_Defense_EV", "Speed_EV",
]:
    pokemon_df[column] = pd.to_numeric(pokemon_df[column], errors="coerce")

print("Rows:", len(pokemon_df))
print("Unique names:", pokemon_df["Pokemon"].nunique())
print("Missing primary types:", (pokemon_df["Type_I"].fillna("").str.strip() == "").sum())
print("Missing secondary types:", (pokemon_df["Type II"].fillna("").str.strip() == "").sum())
print("Missing heights:", pokemon_df["Height_m"].isna().sum())
print("Missing weights:", pokemon_df["Weight_kg"].isna().sum())
print("Missing EVs:", pokemon_df["HP_EV"].isna().sum())

display(pokemon_df.head(15))
display(pokemon_df[
    ["Pokemon", "Type_I", "Type II", "Height_m", "Weight_kg"]
].head(25))

Rows: 1025
Unique names: 1025
Missing primary types: 0
Missing secondary types: 489
Missing heights: 0
Missing weights: 0
Missing EVs: 0


,Pokemon,Generation,HP,Attack,Defense,Special_Attack,Special_Defense,Speed,hp_over_total,attack_over_total,...,Hidden_Ability,Height_m,Weight_kg,Height_over_weight,HP_EV,Attcak_EV,Defense_EV,Special_Attack_EV,Special_Defense_EV,Speed_EV
0,Bulbasaur,1,45,49,49,65,65,45,0.141509,0.154088,...,,0.7,6.9,0.101449,0,0,0,1,0,0
1,Ivysaur,1,60,62,63,80,80,60,0.148148,0.153086,...,,1.0,13.0,0.076923,0,0,0,1,1,0
2,Venusaur,1,80,82,83,100,100,80,0.152381,0.156190,...,,2.0,100.0,0.020000,0,0,0,2,1,0
3,Charmander,1,39,52,43,60,50,65,0.126214,0.168285,...,,0.6,8.5,0.070588,0,0,0,0,0,1
4,Charmeleon,1,58,64,58,80,65,80,0.143210,0.158025,...,,1.1,19.0,0.057895,0,0,0,1,0,1
5,Charizard,1,78,84,78,109,85,100,0.146067,0.157303,...,,1.7,90.5,0.018785,0,0,0,3,0,0
6,Squirtle,1,44,48,65,50,64,43,0.140127,0.152866,...,,0.5,9.0,0.055556,0,0,1,0,0,0
7,Wartortle,1,59,63,80,65,80,58,0.145679,0.155556,...,,1.0,22.5,0.044444,0,0,1,0,1,0
8,Blastoise,1,79,83,100,85,105,78,0.149057,0.156604,...,,1.6,85.5,0.018713,0,0,0,0,3,0
9,Caterpie,1,45,30,35,20,20,45,0.230769,0.153846,...,,0.3,2.9,0.103448,1,0,0,0,0,0


,Pokemon,Type_I,Type II,Height_m,Weight_kg
0,Bulbasaur,Grass,Poison,0.7,6.9
1,Ivysaur,Grass,Poison,1.0,13.0
2,Venusaur,Grass,Poison,2.0,100.0
3,Charmander,Fire,,0.6,8.5
4,Charmeleon,Fire,,1.1,19.0
5,Charizard,Fire,Flying,1.7,90.5
6,Squirtle,Water,,0.5,9.0
7,Wartortle,Water,,1.0,22.5
8,Blastoise,Water,,1.6,85.5
9,Caterpie,Bug,,0.3,2.9


In [7]:
# Export the CSV.
pokemon_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print(f"Saved {len(pokemon_df)} rows to: {OUTPUT_FILE.resolve()}")

# Read the file back to confirm it was written successfully.
verification_df = pd.read_csv(OUTPUT_FILE)
print("Verified CSV rows:", len(verification_df))
print("Verified CSV columns:", len(verification_df.columns))
display(verification_df.head())

Saved 1025 rows to: C:\Users\tykur\pokedex_serebii.csv
Verified CSV rows: 1025
Verified CSV columns: 29


,Pokemon,Generation,HP,Attack,Defense,Special_Attack,Special_Defense,Speed,hp_over_total,attack_over_total,...,Hidden_Ability,Height_m,Weight_kg,Height_over_weight,HP_EV,Attcak_EV,Defense_EV,Special_Attack_EV,Special_Defense_EV,Speed_EV
0,Bulbasaur,1,45,49,49,65,65,45,0.141509,0.154088,...,NaN,0.7,6.9,0.101449,0,0,0,1,0,0
1,Ivysaur,1,60,62,63,80,80,60,0.148148,0.153086,...,NaN,1.0,13.0,0.076923,0,0,0,1,1,0
2,Venusaur,1,80,82,83,100,100,80,0.152381,0.156190,...,NaN,2.0,100.0,0.020000,0,0,0,2,1,0
3,Charmander,1,39,52,43,60,50,65,0.126214,0.168285,...,NaN,0.6,8.5,0.070588,0,0,0,0,0,1
4,Charmeleon,1,58,64,58,80,65,80,0.143210,0.158025,...,NaN,1.1,19.0,0.057895,0,0,0,1,0,1
